In [52]:
from ultralytics import YOLO
import os
from tqdm import tqdm
import pandas as pd

In [ ]:
import numpy as np
from pathlib import Path

def calculate_iou(box1, box2):
    """計算兩個邊界框的 IoU
    box format: [x1, y1, x2, y2]
    """
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2

    # 計算交集區域
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)

    inter_area = max(0, inter_x_max - inter_x_min) * max(0, inter_y_max - inter_y_min)

    # 計算聯集區域
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area if union_area > 0 else 0
    return iou

def yolo_to_xyxy(yolo_box, img_width, img_height):
    """將 YOLO 格式 (center_x, center_y, w, h) 轉換為 (x1, y1, x2, y2)
    yolo_box: 歸一化的 [center_x, center_y, width, height]
    """
    center_x, center_y, w, h = yolo_box
    x1 = (center_x - w/2) * img_width
    y1 = (center_y - h/2) * img_height
    x2 = (center_x + w/2) * img_width
    y2 = (center_y + h/2) * img_height
    return [x1, y1, x2, y2]

def box_center(box):
    """取得 xyxy box 的中心座標"""
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])

def calculate_recall_precision(result, image_path, iou_threshold=0.5):
    """計算單張圖片的 recall 和 precision
    只計算 nodule (class 0)
    - 預測 nodule 與 GT nodule 重疊 -> TP
    - 預測 nodule 與 GT maybe_nodule 重疊 -> 忽略（不算 TP 也不算 FP）
    - 預測 nodule 都沒重疊 -> FP
    """

    # 獲取圖片尺寸
    img_height, img_width = result.orig_shape

    # 獲取預測框 (xyxy format) 和類別
    pred_boxes = []
    pred_classes = []
    if result.boxes is not None and len(result.boxes) > 0:
        pred_boxes = result.boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
        pred_classes = result.boxes.cls.cpu().numpy()  # class ids

    # 讀取 ground truth 標籤
    label_path = image_path.replace('/images/', '/labels/').replace('.png', '.txt').replace('.jpg', '.txt')
    gt_boxes_nodule = []
    gt_boxes_maybe_nodule = []

    if Path(label_path).exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    # YOLO 格式: class_id center_x center_y width height
                    class_id, cx, cy, w, h = map(float, parts[:5])
                    xyxy = yolo_to_xyxy([cx, cy, w, h], img_width, img_height)
                    if class_id == 0:  # nodule
                        gt_boxes_nodule.append(xyxy)
                    elif class_id == 1:  # maybe_nodule
                        gt_boxes_maybe_nodule.append(xyxy)

    # 只計算 nodule (class 0) 的指標
    num_gt = len(gt_boxes_nodule)
    # 只統計預測為 nodule (class 0) 的框
    pred_nodule_indices = [i for i, cls in enumerate(pred_classes) if cls == 0]
    num_pred = len(pred_nodule_indices)

    if num_gt == 0 and num_pred == 0:
        return {'recall': 1.0, 'precision': 1.0, 'TP': 0, 'FP': 0, 'FN': 0}

    if num_gt == 0:
        return {'recall': 0.0, 'precision': 0.0, 'TP': 0, 'FP': num_pred, 'FN': 0}

    if num_pred == 0:
        return {'recall': 0.0, 'precision': 0.0, 'TP': 0, 'FP': 0, 'FN': num_gt}

    # 匹配預測框和真實框
    matched_gt = set()
    matched_pred = set()
    matched_pairs = []  # (pred_idx, gt_idx)
    ignored_pred = set()  # 與 maybe_nodule 重疊的預測框

    for i in pred_nodule_indices:
        pred_box = pred_boxes[i]
        best_iou = 0
        best_gt_idx = -1

        # 先嘗試與 GT 中的 nodule (class 0) 匹配
        for j, gt_box in enumerate(gt_boxes_nodule):
            if j in matched_gt:
                continue

            iou = calculate_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = j

        # 如果與 nodule GT 的 IoU >= 閾值，則為 TP
        if best_iou >= iou_threshold:
            matched_gt.add(best_gt_idx)
            matched_pred.add(i)
            matched_pairs.append((i, best_gt_idx))
        else:
            # 如果與 nodule 沒有足夠重疊，檢查是否與 maybe_nodule 重疊
            max_maybe_iou = 0
            for gt_box in gt_boxes_maybe_nodule:
                iou = calculate_iou(pred_box, gt_box)
                if iou > max_maybe_iou:
                    max_maybe_iou = iou
            
            # 如果與 maybe_nodule 的 IoU >= 閾值，則忽略此預測框
            if max_maybe_iou >= iou_threshold:
                ignored_pred.add(i)
            # 否則算 FP（在下面的 FP 計算中處理）

    TP = len(matched_gt)  # 正確檢測到的 nodule 數量
    # FP = 預測框中既沒有與 nodule 匹配，也沒有被忽略的數量
    FP = num_pred - len(matched_pred) - len(ignored_pred)
    FN = num_gt - len(matched_gt)  # 沒有被檢測到的 nodule 數量

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0

    return {
        'recall': recall,
        'precision': precision,
        'TP': TP,
        'FP': FP,
        'FN': FN,
        'center_distances': [
            np.linalg.norm(box_center(pred_boxes[i]) - box_center(gt_boxes_nodule[j]))
            for i, j in matched_pairs
        ],
        'num_gt': num_gt,
        'num_pred': num_pred,
        'num_ignored': len(ignored_pred)  # 額外返回被忽略的預測框數量
    }

In [54]:
def get_patient_metrics(df):
    patient_metrics = df.groupby('patient_ID').agg({
        'TP': 'sum',
        'FP': 'sum',
        'FN': 'sum'
    }).reset_index()

    eta = 1e-9
    patient_metrics['recall'] = patient_metrics['TP'] / (patient_metrics['TP'] + patient_metrics['FN'] + eta)
    patient_metrics['precision'] = patient_metrics['TP'] / (patient_metrics['TP'] + patient_metrics['FP'] + eta)

    patient_metrics['f1_score'] = 2 * (patient_metrics['precision'] * patient_metrics['recall']) / (patient_metrics['precision'] + patient_metrics['recall'] + eta)
    patient_mean_recall = patient_metrics['recall'].mean()
    patient_mean_precision = patient_metrics['precision'].mean()
    patient_mean_f1 = patient_metrics['f1_score'].mean()
    return patient_mean_recall, patient_mean_precision, patient_mean_f1

In [55]:
def get_file_metrics(df):
    file_metrics = df.groupby('file_name').agg({
        'TP': 'sum',
        'FP': 'sum',
        'FN': 'sum'
    }).reset_index()

    eta = 1e-9
    file_metrics['recall'] = file_metrics['TP'].sum() / (file_metrics['TP'].sum() + file_metrics['FN'].sum() + eta)
    file_metrics['precision'] = file_metrics['TP'].sum() / (file_metrics['TP'].sum() + file_metrics['FP'].sum() + eta)
    file_metrics['f1_score'] = 2 * (file_metrics['precision'] * file_metrics['recall']) / (file_metrics['precision'] + file_metrics['recall'] + eta)
    return file_metrics['recall'].iloc[0], file_metrics['precision'].iloc[0], file_metrics['f1_score'].iloc[0]

In [57]:
name = "2026_02_11(2)"  # 模型路徑
model_path = f'../yolo_output/runs/detect/{name}/weights/best.pt'

model = YOLO(model_path)

In [58]:
performance_summary = {
    "IOU_0.3": {
        "patient_mean_recall": None,
        "patient_mean_precision": None,
        "patient_mean_f1": None,
        "file_mean_recall": None,
        "file_mean_precision": None,
        "file_mean_f1": None
    },
    "IOU_0.5": {
        "patient_mean_recall": None,
        "patient_mean_precision": None,
        "patient_mean_f1": None,
        "file_mean_recall": None,
        "file_mean_precision": None,
        "file_mean_f1": None
    },
    "IOU_0.7": {
        "patient_mean_recall": None,
        "patient_mean_precision": None,
        "patient_mean_f1": None,
        "file_mean_recall": None,
        "file_mean_precision": None,
        "file_mean_f1": None
    }
}

In [59]:
data_df = pd.read_csv("../yolo_data/all_csv_files/combined_dataset_info_v1.csv")
val_CG_df = data_df[(data_df['dataset'] == "CG") &  (data_df['train_or_test'] == "test")]
val_CG_single_df = data_df[(data_df['dataset'] == "CG") & (data_df["nodule_type"]=="single") & (data_df['train_or_test'] == "test")]

In [60]:
data_root = "../yolo_data/all_data_nodule/images"
val_df = val_CG_single_df

file_name_list  = [os.path.join(data_root, f"{file_name}.png") for file_name in val_df["file_name"].to_list()]
patient_ID_list = val_df["patient_ID"].to_list()
conf_threshold = 0.25

# Process images in batches to avoid memory overflow
batch_size = 32  # Adjust this based on your available memory

for iou_threshold in [0.3, 0.5, 0.7]:
    all_result = {
        'patient_ID': [],
        'file_name': [],
        'TP': [],
        'FP': [],
        'FN': []
    }
    for i in tqdm(range(0, len(file_name_list), batch_size), desc="Processing batches"):
        batch_file_names = file_name_list[i:i+batch_size]
        batch_patient_IDs = patient_ID_list[i:i+batch_size]
        results = model(batch_file_names, conf=conf_threshold)

        for image_path, result in zip(batch_file_names, results):
            img_name = Path(image_path).name
            metrics = calculate_recall_precision(result, image_path, iou_threshold=iou_threshold)

            all_result['patient_ID'].append(img_name)
            all_result['file_name'].append(img_name)
            all_result['TP'].append(metrics['TP'])
            all_result['FP'].append(metrics['FP'])
            all_result['FN'].append(metrics['FN'])
    all_result_df = pd.DataFrame(all_result)
    patient_mean_recall, patient_mean_precision, patient_mean_f1 = get_patient_metrics(all_result_df)
    file_mean_recall, file_mean_precision, file_mean_f1 = get_file_metrics(all_result_df)
    performance_summary[f"IOU_{iou_threshold}"]["patient_mean_recall"] = patient_mean_recall
    performance_summary[f"IOU_{iou_threshold}"]["patient_mean_precision"] = patient_mean_precision   
    performance_summary[f"IOU_{iou_threshold}"]["patient_mean_f1"] = patient_mean_f1
    performance_summary[f"IOU_{iou_threshold}"]["file_mean_recall"] = file_mean_recall
    performance_summary[f"IOU_{iou_threshold}"]["file_mean_precision"] = file_mean_precision
    performance_summary[f"IOU_{iou_threshold}"]["file_mean_f1"] = file_mean_f1

performance_df = pd.DataFrame(performance_summary).T


Processing batches:   0%|          | 0/9 [00:00<?, ?it/s]


0: 640x640 1 nodule, 1.1ms
1: 640x640 (no detections), 1.1ms
2: 640x640 (no detections), 1.1ms
3: 640x640 2 nodules, 1.1ms
4: 640x640 1 nodule, 1.1ms
5: 640x640 1 nodule, 1.1ms
6: 640x640 2 nodules, 1.1ms
7: 640x640 1 nodule, 1.1ms
8: 640x640 1 nodule, 1.1ms
9: 640x640 1 nodule, 1.1ms
10: 640x640 1 nodule, 1.1ms
11: 640x640 1 nodule, 1.1ms
12: 640x640 1 nodule, 1.1ms
13: 640x640 1 nodule, 1.1ms
14: 640x640 1 nodule, 1.1ms
15: 640x640 (no detections), 1.1ms
16: 640x640 1 nodule, 1.1ms
17: 640x640 1 nodule, 1.1ms
18: 640x640 1 nodule, 1.1ms
19: 640x640 1 nodule, 1.1ms
20: 640x640 (no detections), 1.1ms
21: 640x640 2 nodules, 1.1ms
22: 640x640 1 nodule, 1.1ms
23: 640x640 1 nodule, 1.1ms
24: 640x640 1 nodule, 1.1ms
25: 640x640 1 nodule, 1.1ms
26: 640x640 1 nodule, 1.1ms
27: 640x640 1 nodule, 1.1ms
28: 640x640 (no detections), 1.1ms
29: 640x640 2 nodules, 1.1ms
30: 640x640 1 nodule, 1.1ms
31: 640x640 1 nodule, 1.1ms
Speed: 2.7ms preprocess, 1.1ms inference, 0.2ms postprocess per image at s

Processing batches:  11%|█         | 1/9 [00:00<00:03,  2.08it/s]


0: 640x640 1 nodule, 1.1ms
1: 640x640 1 nodule, 1.1ms
2: 640x640 1 nodule, 1.1ms
3: 640x640 1 nodule, 1.1ms
4: 640x640 1 nodule, 1.1ms
5: 640x640 1 nodule, 1.1ms
6: 640x640 1 nodule, 1.1ms
7: 640x640 1 nodule, 1.1ms
8: 640x640 1 nodule, 1.1ms
9: 640x640 1 nodule, 1.1ms
10: 640x640 (no detections), 1.1ms
11: 640x640 1 nodule, 1.1ms
12: 640x640 1 nodule, 1.1ms
13: 640x640 1 nodule, 1.1ms
14: 640x640 1 nodule, 1.1ms
15: 640x640 2 nodules, 1.1ms
16: 640x640 1 nodule, 1.1ms
17: 640x640 1 nodule, 1.1ms
18: 640x640 1 nodule, 1.1ms
19: 640x640 1 nodule, 1.1ms
20: 640x640 (no detections), 1.1ms
21: 640x640 1 nodule, 1.1ms
22: 640x640 1 nodule, 1.1ms
23: 640x640 1 nodule, 1.1ms
24: 640x640 1 nodule, 1.1ms
25: 640x640 2 nodules, 1.1ms
26: 640x640 1 nodule, 1.1ms
27: 640x640 1 nodule, 1.1ms
28: 640x640 1 nodule, 1.1ms
29: 640x640 1 nodule, 1.1ms
30: 640x640 (no detections), 1.1ms
31: 640x640 1 nodule, 1.1ms
Speed: 2.7ms preprocess, 1.1ms inference, 0.2ms postprocess per image at shape (1, 3, 640,

Processing batches:  22%|██▏       | 2/9 [00:00<00:03,  2.27it/s]

...

Processing batches:  78%|███████▊  | 7/9 [00:02<00:00,  2.48it/s]


0: 640x640 2 nodules, 1.1ms
1: 640x640 1 nodule, 1.1ms
2: 640x640 1 nodule, 1.1ms
3: 640x640 1 nodule, 1.1ms
4: 640x640 1 nodule, 1.1ms
5: 640x640 1 nodule, 1.1ms
6: 640x640 1 nodule, 1.1ms
7: 640x640 2 nodules, 1.1ms
8: 640x640 1 nodule, 1.1ms
9: 640x640 1 nodule, 1.1ms
10: 640x640 1 nodule, 1.1ms
11: 640x640 1 nodule, 1.1ms
12: 640x640 1 nodule, 1.1ms
13: 640x640 1 nodule, 1.1ms
14: 640x640 1 nodule, 1.1ms
15: 640x640 (no detections), 1.1ms
16: 640x640 1 nodule, 1.1ms
17: 640x640 1 nodule, 1.1ms
18: 640x640 1 nodule, 1.1ms
19: 640x640 1 nodule, 1.1ms
20: 640x640 1 nodule, 1.1ms
21: 640x640 1 nodule, 1.1ms
22: 640x640 1 nodule, 1.1ms
23: 640x640 1 nodule, 1.1ms
24: 640x640 1 nodule, 1.1ms
25: 640x640 1 nodule, 1.1ms
26: 640x640 1 nodule, 1.1ms
27: 640x640 1 nodule, 1.1ms
28: 640x640 1 nodule, 1.1ms
29: 640x640 1 nodule, 1.1ms
30: 640x640 (no detections), 1.1ms
31: 640x640 1 nodule, 1.1ms
Speed: 2.6ms preprocess, 1.1ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)


Processing batches:  89%|████████▉ | 8/9 [00:03<00:00,  2.43it/s]


0: 640x640 (no detections), 1.2ms
1: 640x640 1 nodule, 1.2ms
2: 640x640 1 nodule, 1.2ms
3: 640x640 (no detections), 1.2ms
4: 640x640 1 nodule, 1.2ms
5: 640x640 1 nodule, 1.2ms
6: 640x640 1 nodule, 1.2ms
7: 640x640 (no detections), 1.2ms
8: 640x640 1 nodule, 1.2ms
9: 640x640 1 nodule, 1.2ms
10: 640x640 1 nodule, 1.2ms
11: 640x640 1 nodule, 1.2ms
12: 640x640 1 nodule, 1.2ms
13: 640x640 1 nodule, 1.2ms
14: 640x640 1 nodule, 1.2ms
15: 640x640 1 nodule, 1.2ms
16: 640x640 1 nodule, 1.2ms
17: 640x640 1 nodule, 1.2ms
18: 640x640 1 nodule, 1.2ms
19: 640x640 1 nodule, 1.2ms
20: 640x640 1 nodule, 1.2ms
21: 640x640 1 nodule, 1.2ms
22: 640x640 1 nodule, 1.2ms
23: 640x640 1 nodule, 1.2ms
24: 640x640 1 nodule, 1.2ms
25: 640x640 1 nodule, 1.2ms
Speed: 2.4ms preprocess, 1.2ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)


Processing batches: 100%|██████████| 9/9 [00:03<00:00,  2.57it/s]


In [61]:
performance_df

,patient_mean_recall,patient_mean_precision,patient_mean_f1,file_mean_recall,file_mean_precision,file_mean_f1
IOU_0.3,0.861702,0.831560,0.841608,0.861702,0.896679,0.878843
IOU_0.5,0.836879,0.808511,0.817967,0.836879,0.870849,0.853526
IOU_0.7,0.755319,0.730496,0.738771,0.755319,0.785978,0.770344


In [ ]:
## OLD version
# data_root = "../yolo_data/nodule_not_clean_yolo/val/images"
# image_dir = os.listdir(data_root)

# image_dir  = [os.path.join(data_root, img) for img in image_dir]
# conf_threshold = 0.25


# # Process images in batches to avoid memory overflow
# batch_size = 32  # Adjust this based on your available memory

# for iou_threshold in [0.3, 0.5, 0.7]:
#     all_result = {
#         'patient_ID': [],
#         'file_name': [],
#         'TP': [],
#         'FP': [],
#         'FN': []
#     }
#     for i in tqdm(range(0, len(image_dir), batch_size), desc="Processing batches"):
#         batch_images = image_dir[i:i+batch_size]
#         results = model(batch_images, conf=conf_threshold)

#         for image_path, result in zip(batch_images, results):
#             img_name = Path(image_path).name
#             metrics = calculate_recall_precision(result, image_path, iou_threshold=iou_threshold)

#             all_result['patient_ID'].append(img_name.split('_')[0])
#             all_result['file_name'].append(img_name.split('_')[1])
#             all_result['TP'].append(metrics['TP'])
#             all_result['FP'].append(metrics['FP'])
#             all_result['FN'].append(metrics['FN'])
#     all_result_df = pd.DataFrame(all_result)
#     patient_mean_recall, patient_mean_precision, patient_mean_f1 = get_patient_metrics(all_result_df)
#     file_mean_recall, file_mean_precision, file_mean_f1 = get_file_metrics(all_result_df)
#     performance_summary[f"IOU_{iou_threshold}"]["patient_mean_recall"] = patient_mean_recall
#     performance_summary[f"IOU_{iou_threshold}"]["patient_mean_precision"] = patient_mean_precision   
#     performance_summary[f"IOU_{iou_threshold}"]["patient_mean_f1"] = patient_mean_f1
#     performance_summary[f"IOU_{iou_threshold}"]["file_mean_recall"] = file_mean_recall
#     performance_summary[f"IOU_{iou_threshold}"]["file_mean_precision"] = file_mean_precision
#     performance_summary[f"IOU_{iou_threshold}"]["file_mean_f1"] = file_mean_f1

# performance_df = pd.DataFrame(performance_summary).T


In [73]:
def get_patient_ID(data_folder):
    """從圖片路徑中提取 patient_ID"""
    image_dir = os.listdir(data_folder)
    patient_ids = [img.split('_')[0] for img in image_dir]
    return patient_ids

In [82]:
def have_same_patient_ID(train_ID, val_ID):
    """檢查兩個資料夾中的圖片是否有相同的 patient_ID"""
    patient_ids1 = set(train_ID)
    patient_ids2 = set(val_ID)
    common_ids = patient_ids1.intersection(patient_ids2)
    return len(common_ids) > 0, common_ids

In [86]:
train_dir_path = "../yolo_data/nodule_not_clean_yolo/train/images"
val_dir_path = "../yolo_data/nodule_not_clean_yolo/val/images"
train_ID = get_patient_ID(train_dir_path)
val_ID = get_patient_ID(val_dir_path)
print("len(train_ID):", len(set(train_ID)))
print("len(val_ID):", len(set(val_ID)))
print("len train_ID sample:", len(train_ID))
print("len val_ID sample:", len(val_ID))
common_ids, common_ids_list = have_same_patient_ID(train_ID, val_ID)
print(common_ids, common_ids_list)

len(train_ID): 1690
len(val_ID): 423
len train_ID sample: 6090
len val_ID sample: 1578
False set()
